# Séance 2 — Sortir de PyTorch : export ONNX, mesures et optimisation du graphe

*Cours « Compression de modèles pour l'IA embarquée » — séance 2/7 — 3 h*

Votre U-Net fonctionne dans PyTorch. Une cible embarquée n'exécute pas PyTorch : elle exécute un **graphe de calcul** avec un moteur d'inférence léger. Aujourd'hui, vous exportez votre modèle en ONNX, vous apprenez à le **mesurer** correctement sur CPU, puis vous appliquez les premières optimisations, celles qui ne coûtent aucune précision.

**À la fin de la séance, vous saurez :**
- exporter un modèle PyTorch en ONNX et lire le graphe obtenu ;
- mesurer les quatre grandeurs du cours : taille, latence, mémoire vive, mIoU ;
- fusionner à la main une convolution et sa BatchNorm, et expliquer pourquoi c'est exact ;
- dire ce que fait chaque niveau d'optimisation d'ONNX Runtime.

| Partie | Contenu | Durée |
|---|---|---|
| 0 | Mise en place | 10 min |
| 1 | Exporter en ONNX et lire le graphe | 35 min |
| 2 | Mesurer : taille, latence, mémoire, précision | 45 min |
| 3 | Fusion convolution + BatchNorm | 45 min |
| 4 | Les optimisations d'ONNX Runtime | 30 min |
| 5 | Fiche de référence | 15 min |

**Mode d'emploi.** Comme en séance 1, chaque cellule de code porte une étiquette sur sa première ligne :

| Étiquette | Ce que vous faites |
|---|---|
| `# ✏️ À COMPLÉTER` | Vous écrivez du code : chaque `A_COMPLETER()` est à remplacer, en suivant la consigne placée juste au-dessus. |
| `# ✅ VÉRIFICATION` | Vous exécutez sans modifier : la cellule vous dit si vous pouvez continuer. |
| `# ▶ FOURNI` | Vous exécutez sans modifier. |

## 0. Mise en place

**Pas de GPU aujourd'hui** : toutes les mesures se font sur CPU. Vous pouvez laisser le type d'exécution par défaut de Colab.

Il vous faut le fichier `unet_camvid_base32.pt` enregistré à la fin de la séance 1.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
!pip install -q onnx onnxruntime

import collections, copy, glob, json, os, subprocess, sys, time, warnings

import numpy as np
import onnx
import onnxruntime as ort
import torch
import torch.nn as nn
from PIL import Image

warnings.filterwarnings("ignore")          # l'exportateur ONNX est bavard
print("PyTorch", torch.__version__, "| ONNX Runtime", ort.__version__, "| cœurs CPU :", os.cpu_count())


def A_COMPLETER():
    """Marque le code que vous devez écrire. À remplacer partout où il apparaît."""
    raise NotImplementedError("✏️ Code à compléter : remplacez A_COMPLETER() par votre code "
                              "(la consigne est juste au-dessus de la ligne signalée).")

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
# Le U-Net de la séance 1, à l'identique
class DoubleConv(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, n_classes=11, base=32):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]
        self.enc1 = DoubleConv(3, c[0])
        self.enc2 = DoubleConv(c[0], c[1])
        self.enc3 = DoubleConv(c[1], c[2])
        self.enc4 = DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[4])
        self.up4 = nn.ConvTranspose2d(c[4], c[3], kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c[4], c[3])
        self.up3 = nn.ConvTranspose2d(c[3], c[2], kernel_size=2, stride=2)
        self.dec3 = DoubleConv(c[3], c[2])
        self.up2 = nn.ConvTranspose2d(c[2], c[1], kernel_size=2, stride=2)
        self.dec2 = DoubleConv(c[2], c[1])
        self.up1 = nn.ConvTranspose2d(c[1], c[0], kernel_size=2, stride=2)
        self.dec1 = DoubleConv(c[1], c[0])
        self.head = nn.Conv2d(c[0], n_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
# Chargement du modèle entraîné en séance 1
CKPT = "unet_camvid_base32.pt"
if not os.path.exists(CKPT):
    try:
        from google.colab import files
        print("Sélectionnez votre fichier unet_camvid_base32.pt (séance 1)")
        files.upload()
    except ImportError:
        pass
assert os.path.exists(CKPT), "Fichier unet_camvid_base32.pt introuvable : déposez-le dans l'espace de travail."

ckpt = torch.load(CKPT, map_location="cpu")
N_CLASSES, IGNORE = ckpt["n_classes"], 11
H, W = ckpt["input_size"]

model = UNet(N_CLASSES, base=ckpt["base"])
model.load_state_dict(ckpt["state_dict"])
model.eval()          # indispensable : toute la séance concerne l'inférence
print(f"Modèle chargé : base={ckpt['base']}, entrée {H}×{W}, mIoU de test en séance 1 : {100 * ckpt['test_miou']:.1f} %")

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
# Jeu de test CamVid (233 images), prétraité comme en séance 1
if not os.path.exists("SegNet-Tutorial"):
    !git clone -q --depth 1 https://github.com/alexgkendall/SegNet-Tutorial.git
ROOT = "SegNet-Tutorial/CamVid"

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_test():
    """Retourne les images normalisées float32 (N, 3, H, W) et les masques int64 (N, H, W)."""
    xs, ys = [], []
    for p in sorted(glob.glob(f"{ROOT}/test/*.png")):
        img = Image.open(p).convert("RGB").resize((W, H), Image.BILINEAR)
        mask = Image.open(p.replace("/test/", "/testannot/")).resize((W, H), Image.NEAREST)
        xs.append((np.asarray(img, dtype=np.float32) / 255 - MEAN) / STD)
        ys.append(np.asarray(mask))
    x = np.ascontiguousarray(np.stack(xs).transpose(0, 3, 1, 2), dtype=np.float32)
    return x, np.stack(ys).astype(np.int64)


test_x, test_y = load_test()
print("Test :", test_x.shape, test_x.dtype, "|", test_y.shape)

## 1. Exporter en ONNX et lire le graphe

**ONNX** est un format de fichier qui décrit un réseau comme un **graphe d'opérateurs standard** (`Conv`, `Relu`, `MaxPool`…) accompagné de ses poids. Le fichier ne contient plus de code Python : n'importe quel moteur compatible peut l'exécuter. Nous utiliserons **ONNX Runtime** sur CPU ; le même fichier pourra plus tard être donné à TensorRT sur GPU.

```
PyTorch (entraînement)  ──export──►  fichier .onnx  ──►  ONNX Runtime (CPU)
                                                    ──►  TensorRT (GPU), OpenVINO, …
```

L'export fonctionne par **traçage** : PyTorch exécute le modèle sur un exemple d'entrée et enregistre les opérations rencontrées. Il faut donc lui fournir une entrée de la bonne forme.

### Exercice 1.1 — L'export

Complétez `export_onnx` avec un appel à `torch.onnx.export`. Les arguments à passer :
- le modèle, puis l'entrée d'exemple sous forme de tuple `(x,)`, puis le chemin du fichier ;
- `input_names=["image"]` et `output_names=["logits"]` ;
- `opset_version=17` (la version du jeu d'opérateurs ONNX) ;
- `dynamo=False` (l'exportateur historique, qui écrit un seul fichier) ;
- `do_constant_folding=optimiser`.

Pour ce premier export, nous demandons `optimiser=False` : le graphe **brut**, tel que PyTorch le décrit, sans aucune simplification. Vous verrez en partie 3 ce que l'exportateur fait quand on le laisse optimiser.

La forme d'entrée est **fixe** (1 × 3 × 240 × 320). Sur une cible embarquée, on connaît la résolution de la caméra : une forme fixe permet au moteur d'optimiser davantage.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def export_onnx(model, path, optimiser=True):
    """Exporte le modèle au format ONNX, pour une image 1 × 3 × H × W."""
    model = copy.deepcopy(model).eval()
    x = torch.randn(1, 3, H, W)
    # ✏️ À COMPLÉTER : appel à torch.onnx.export
    A_COMPLETER()
    return path


export_onnx(model, "unet_brut.onnx", optimiser=False)

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
onnx.checker.check_model("unet_brut.onnx")
_g = onnx.load("unet_brut.onnx").graph
assert [i.name for i in _g.input] == ["image"] and [o.name for o in _g.output] == ["logits"], "Noms d'entrée/sortie inattendus."
assert [d.dim_value for d in _g.input[0].type.tensor_type.shape.dim] == [1, 3, H, W]
print(f"OK : unet_brut.onnx, {os.path.getsize('unet_brut.onnx') / 1e6:.1f} Mo")

### Exercice 1.2 — Lire le graphe

Un fichier ONNX se lit en Python : `onnx.load(path).graph.node` est la liste des nœuds du graphe, et chaque nœud a un attribut `op_type` (le nom de l'opérateur).

Écrivez `count_ops`, qui retourne le nombre de nœuds de chaque type. Indication : `collections.Counter`.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def count_ops(path):
    """Retourne un dictionnaire {type d'opérateur: nombre de nœuds} pour un fichier ONNX."""
    # ✏️ À COMPLÉTER : charger le graphe et compter les op_type
    A_COMPLETER()


count_ops("unet_brut.onnx")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
_ops = count_ops("unet_brut.onnx")
assert _ops.get("Conv") == 19 and _ops.get("BatchNormalization") == 18, _ops
print("OK :", sum(_ops.values()), "nœuds")

**À faire.** Téléchargez `unet_brut.onnx` (panneau *Fichiers* de Colab) et ouvrez-le dans **Netron** : [netron.app](https://netron.app). Suivez le graphe de l'entrée à la sortie et cliquez sur quelques nœuds.

**Questions.**
1. Retrouvez dans le graphe : les blocs `DoubleConv`, les quatre connexions de saut, la tête. Pourquoi 19 `Conv` et 18 `BatchNormalization` ?
2. Cliquez sur un nœud `Conv` : où sont stockés ses poids ? Quelle est leur forme ?
3. Qu'est-ce qui a disparu par rapport au code PyTorch ?

### Exercice 1.3 — Exécuter le graphe avec ONNX Runtime

Une **session** ONNX Runtime charge le fichier, prépare le graphe et l'exécute. La fonction `make_session` est fournie ; ses options seront expliquées au fur et à mesure.

L'inférence s'écrit `session.run(None, {"image": x})`, où `x` est un tableau NumPy `float32`. Le résultat est une liste contenant une valeur par sortie du graphe.

Exécutez le modèle ONNX sur la première image de test, puis comparez avec la sortie de PyTorch sur la même image.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
NIVEAUX = {
    "aucune": ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
    "basic": ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
    "extended": ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,
    "all": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
}


def make_session(path, niveau="all", threads=1, save_as=None):
    """Crée une session ONNX Runtime sur CPU.

    niveau  : niveau d'optimisation du graphe (voir NIVEAUX)
    threads : nombre de fils d'exécution utilisés pour le calcul
    save_as : si fourni, enregistre dans ce fichier le graphe après optimisation"""
    options = ort.SessionOptions()
    options.graph_optimization_level = NIVEAUX[niveau]
    options.intra_op_num_threads = threads
    if save_as:
        options.optimized_model_filepath = save_as
    return ort.InferenceSession(path, options, providers=["CPUExecutionProvider"])

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
session = make_session("unet_brut.onnx", niveau="aucune")
x = test_x[:1]                                   # (1, 3, H, W), float32

# ✏️ À COMPLÉTER : sortie du modèle ONNX pour x (tableau NumPy de forme (1, 11, H, W))
y_onnx = A_COMPLETER()

# ✏️ À COMPLÉTER : sortie du modèle PyTorch pour le même x, convertie en tableau NumPy
y_torch = A_COMPLETER()

# ✏️ À COMPLÉTER : plus grand écart, en valeur absolue, entre les deux sorties
ecart = A_COMPLETER()
print(f"Écart maximal : {ecart:.2e} (amplitude moyenne des sorties : {np.abs(y_torch).mean():.2f})")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
assert y_onnx.shape == (1, N_CLASSES, H, W), y_onnx.shape
assert ecart < 1e-3, "Les deux sorties diffèrent trop : le modèle est-il bien en mode eval() ?"
assert (y_onnx.argmax(1) == y_torch.argmax(1)).mean() > 0.999
print("OK : mêmes prédictions")

**Question.** L'écart est minuscule, mais pas nul. Pourquoi deux moteurs qui calculent le même graphe avec les mêmes poids ne donnent-ils pas exactement le même résultat ?

## 2. Mesurer : taille, latence, mémoire, précision

On ne compresse pas ce qu'on ne sait pas mesurer. Tout le cours repose sur quatre grandeurs :

| Grandeur | Contrainte embarquée | Unité |
|---|---|---|
| **Taille** du fichier | stockage (mémoire flash), mise à jour à distance | Mo |
| **Latence** | temps réel, énergie | ms par image |
| **Mémoire vive** au pic | RAM de la cible | Mo |
| **Précision** | qualité du service rendu | mIoU |

### Exercice 2.1 — Taille

Le fichier ONNX contient surtout les poids. Calculez la taille attendue à partir du nombre de paramètres du modèle (chaque paramètre est un `float32`) et comparez à la taille réelle du fichier.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : nombre de paramètres du modèle PyTorch
n_params = A_COMPLETER()

# ✏️ À COMPLÉTER : taille attendue en Mo (1 Mo = 1e6 octets)
taille_attendue = A_COMPLETER()

# ✏️ À COMPLÉTER : taille réelle du fichier unet_brut.onnx en Mo (os.path.getsize)
taille_reelle = A_COMPLETER()

print(f"{n_params / 1e6:.2f} M paramètres → {taille_attendue:.2f} Mo attendus, {taille_reelle:.2f} Mo sur le disque")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
assert n_params == 7_763_371
assert abs(taille_attendue - 31.05) < 0.01 and abs(taille_reelle - taille_attendue) < 0.5
print("OK")

**Question.** Le fichier est légèrement plus gros que prévu. Qu'y a-t-il dedans en plus des paramètres appris ? (Pensez à ce que la BatchNorm utilise à l'inférence.)

### Exercice 2.2 — Latence

Mesurer un temps d'exécution demande trois précautions :
1. **Échauffement** : les premières exécutions sont plus lentes (allocations mémoire, caches). On les exécute sans les compter.
2. **Répétition** : une seule mesure ne vaut rien, la machine fait autre chose en même temps.
3. **Statistique robuste** : on rapporte la **médiane**, peu sensible aux mesures aberrantes, et le **95ᵉ centile** (p95), qui décrit les mauvais cas.

Complétez `latency_ms`. Utilisez `time.perf_counter()`, l'horloge la plus précise de Python.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def latency_ms(session, n=20, warmup=3):
    """Retourne (médiane, p95) du temps d'une inférence, en millisecondes."""
    name = session.get_inputs()[0].name
    x = np.random.randn(1, 3, H, W).astype(np.float32)
    # ✏️ À COMPLÉTER : échauffement (warmup inférences non mesurées)
    A_COMPLETER()
    times = []
    for _ in range(n):
        # ✏️ À COMPLÉTER : mesurer une inférence et ajouter sa durée en ms à times
        A_COMPLETER()
    return float(np.median(times)), float(np.percentile(times, 95))


for essai in range(3):
    med, p95 = latency_ms(session)
    print(f"essai {essai + 1} : médiane {med:6.1f} ms | p95 {p95:6.1f} ms")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
_med, _p95 = latency_ms(session, n=5, warmup=1)
assert 1 < _med <= _p95 < 60_000, "Les durées doivent être en millisecondes, avec médiane ≤ p95."
print("OK")

**Questions.**
1. Vos trois essais donnent-ils la même médiane ? De combien varient-ils ? En dessous de quel écart ne peut-on pas dire qu'un modèle est plus rapide qu'un autre ?
2. Pour un système temps réel (une caméra à 10 images par seconde), laquelle des deux statistiques compte le plus ?

### Exercice 2.3 — Nombre de fils d'exécution

ONNX Runtime répartit le calcul d'une convolution sur plusieurs cœurs. Mesurez la latence médiane avec 1 fil, puis avec tous les cœurs de la machine (`os.cpu_count()`).

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
for threads in sorted({1, os.cpu_count()}):
    # ✏️ À COMPLÉTER : créer une session avec ce nombre de fils (niveau "aucune") et mesurer sa latence
    med, p95 = A_COMPLETER()
    print(f"{threads} fil(s) : médiane {med:6.1f} ms")

**Question.** Avec deux fois plus de fils, la latence est-elle divisée par deux ? Pourquoi ?

**Convention du cours.** À partir de maintenant, toutes les latences sont mesurées avec **un seul fil** : la mesure est plus stable, comparable d'une machine à l'autre, et représentative d'une petite cible embarquée.

### Exercice 2.4 — Mémoire vive

La fonction `peak_ram_mb` est fournie. Elle lance un processus Python séparé, y charge le modèle, exécute une inférence, et retourne le **pic de mémoire vive** ajouté par le modèle. Un processus séparé est nécessaire : un pic de mémoire ne redescend pas, on ne peut donc pas mesurer plusieurs modèles dans le même processus.

Avant de mesurer, faites une prédiction. En séance 1, vous avez vu que les plus grosses activations sont en haute résolution. La plus grosse est l'entrée du bloc `dec1` : la concaténation de `up1` (32 canaux) et de `enc1` (32 canaux), en 240 × 320.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
_RAM_SCRIPT = """
import sys, numpy as np, onnxruntime as ort

def pic():   # pic de mémoire résidente du processus, en Mo (Linux)
    for line in open("/proc/self/status"):
        if line.startswith("VmHWM"):
            return int(line.split()[1]) / 1024

avant = pic()
options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.graph_optimization_level = getattr(ort.GraphOptimizationLevel, sys.argv[2])
session = ort.InferenceSession(sys.argv[1], options, providers=["CPUExecutionProvider"])
entree = session.get_inputs()[0]
session.run(None, {entree.name: np.zeros(entree.shape, dtype=np.float32)})
print(pic() - avant)
"""


def peak_ram_mb(path, niveau="all"):
    """Pic de mémoire vive (Mo) ajouté par le chargement du modèle et une inférence."""
    result = subprocess.run([sys.executable, "-c", _RAM_SCRIPT, path, NIVEAUX[niveau].name],
                            capture_output=True, text=True)
    assert result.returncode == 0, result.stderr
    return float(result.stdout.strip())

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : taille en Mo de la plus grosse activation (64 canaux, H × W, float32)
plus_grosse_activation = A_COMPLETER()

# ✏️ À COMPLÉTER : pic de mémoire vive mesuré pour unet_brut.onnx (niveau "aucune")
ram = A_COMPLETER()

print(f"Fichier : {taille_reelle:.0f} Mo | plus grosse activation : {plus_grosse_activation:.1f} Mo | pic mesuré : {ram:.0f} Mo")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
assert abs(plus_grosse_activation - 19.66) < 0.01
assert ram > taille_reelle, "Le pic de mémoire doit au moins contenir les poids."
print("OK")

**Questions.**
1. Le pic de mémoire est nettement plus grand que le fichier. De quoi est-il fait ?
2. Une seule activation pèse déjà les deux tiers de tous les poids du réseau. Pour réduire la mémoire vive, sur quoi faut-il agir : le nombre de paramètres ou autre chose ?

### Exercice 2.5 — Précision

Dernière grandeur : le mIoU, calculé cette fois avec ONNX Runtime sur les 233 images de test. La matrice de confusion (version NumPy de celle de la séance 1) est fournie.

Complétez `miou_onnx`. L'exécution prend environ une minute.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
def confusion_matrix(pred, target):
    keep = target != IGNORE
    index = target[keep] * N_CLASSES + pred[keep]
    return np.bincount(index, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)


def miou_from_cm(cm):
    inter = np.diag(cm)
    union = cm.sum(axis=0) + cm.sum(axis=1) - inter
    return float((inter / np.maximum(union, 1)).mean())

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def miou_onnx(session, x=test_x, y=test_y):
    """mIoU d'une session ONNX Runtime sur le jeu de test, image par image."""
    name = session.get_inputs()[0].name
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    for i in range(len(x)):
        # ✏️ À COMPLÉTER : inférence sur l'image i (attention : le modèle attend la forme (1, 3, H, W))
        logits = A_COMPLETER()
        # ✏️ À COMPLÉTER : classe prédite pour chaque pixel, de forme (H, W)
        pred = A_COMPLETER()
        cm += confusion_matrix(pred, y[i])
    return miou_from_cm(cm)


miou = miou_onnx(session)
print(f"mIoU ONNX Runtime : {100 * miou:.2f} %  |  mIoU PyTorch (séance 1) : {100 * ckpt['test_miou']:.2f} %")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification : l'export ne doit rien changer à la précision
assert abs(miou - ckpt["test_miou"]) < 0.002, "Le mIoU a changé : vérifiez l'inférence et l'argmax."
print("OK : export sans perte")

## 3. Fusion convolution + BatchNorm

Première optimisation du cours, et la seule qui soit entièrement gratuite.

**Pendant l'entraînement**, la BatchNorm normalise avec la moyenne et la variance du lot courant. **À l'inférence**, elle utilise des statistiques figées ($\mu$, $\sigma^2$), calculées pendant l'entraînement. Elle devient alors une simple transformation affine, canal par canal :

$$y = \gamma\,\frac{z - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta$$

Or $z$ est la sortie d'une convolution, $z = W * x + b$, elle aussi affine. Deux transformations affines qui se suivent en font une seule. En posant $s = \dfrac{\gamma}{\sqrt{\sigma^2 + \varepsilon}}$ (un nombre par canal de sortie) :

$$y = \underbrace{(s \cdot W)}_{W'} * x + \underbrace{(b - \mu)\,s + \beta}_{b'}$$

On remplace donc la convolution et sa BatchNorm par **une seule convolution**, de poids $W'$ et de biais $b'$. Le résultat est le même, aux arrondis près.

### Exercice 3.1 — Fusionner une paire

Complétez `fuse_conv_bn`. Les grandeurs utiles :

| Formule | PyTorch |
|---|---|
| $\gamma$, $\beta$ | `bn.weight`, `bn.bias` |
| $\mu$, $\sigma^2$, $\varepsilon$ | `bn.running_mean`, `bn.running_var`, `bn.eps` |
| $W$ | `conv.weight`, de forme `(C_out, C_in, 3, 3)` |
| $b$ | nul ici (`bias=False`) |

Attention aux formes : $s$ contient une valeur par canal de sortie, et doit multiplier chaque filtre de $W$ en entier (`s.view(-1, 1, 1, 1)`).

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
@torch.no_grad()
def fuse_conv_bn(conv, bn):
    """Retourne une convolution équivalente à conv suivie de bn (en mode évaluation)."""
    assert conv.bias is None, "Cette version suppose une convolution sans biais."
    fused = nn.Conv2d(conv.in_channels, conv.out_channels, conv.kernel_size,
                      conv.stride, conv.padding, bias=True)
    # ✏️ À COMPLÉTER : s, un facteur par canal de sortie
    s = A_COMPLETER()
    # ✏️ À COMPLÉTER : nouveaux poids W'
    new_weight = A_COMPLETER()
    # ✏️ À COMPLÉTER : nouveau biais b'
    new_bias = A_COMPLETER()
    fused.weight.copy_(new_weight)
    fused.bias.copy_(new_bias)
    return fused

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification sur une paire aux statistiques aléatoires
torch.manual_seed(0)
_conv, _bn = nn.Conv2d(8, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16).eval()
_bn.running_mean.normal_(); _bn.running_var.uniform_(0.5, 2)
nn.init.uniform_(_bn.weight, 0.5, 1.5); nn.init.normal_(_bn.bias)
_x = torch.randn(2, 8, 20, 24)
with torch.no_grad():
    _ecart = (fuse_conv_bn(_conv, _bn)(_x) - _bn(_conv(_x))).abs().max().item()
assert _ecart < 1e-5, f"Écart {_ecart:.2e} : la convolution fusionnée n'est pas équivalente."
print(f"OK : écart maximal {_ecart:.1e}")

### Exercice 3.2 — Fusionner tout le réseau

Chaque `DoubleConv` contient un `nn.Sequential` de six couches : `conv, bn, relu, conv, bn, relu`. Remplacez-le par un `nn.Sequential` de quatre couches, où chaque paire convolution + BatchNorm est fusionnée.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def fuse_model(model):
    """Retourne une copie du modèle dans laquelle toutes les paires Conv + BatchNorm sont fusionnées."""
    fused = copy.deepcopy(model).eval()
    for module in fused.modules():
        if isinstance(module, DoubleConv):
            conv1, bn1, relu1, conv2, bn2, relu2 = module.block
            # ✏️ À COMPLÉTER : nouveau nn.Sequential de quatre couches
            module.block = A_COMPLETER()
    return fused


model_fused = fuse_model(model)

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
assert not any(isinstance(m, nn.BatchNorm2d) for m in model_fused.modules()), "Il reste des BatchNorm."
with torch.no_grad():
    _x = torch.from_numpy(test_x[:2])
    _ecart = (model_fused(_x) - model(_x)).abs().max().item()
assert _ecart < 1e-3, f"Écart {_ecart:.2e} : le modèle fusionné n'est pas équivalent."
_n = sum(p.numel() for p in model_fused.parameters())
print(f"OK : écart maximal {_ecart:.1e} | {n_params:,d} → {_n:,d} paramètres")

**Questions.**
1. Le nombre de paramètres a baissé de 2 944. Retrouvez ce nombre par le calcul.
2. Peut-on fusionner les BatchNorm **avant** l'entraînement, pour entraîner un réseau plus simple ? Pourquoi ?

### Exercice 3.3 — Le graphe fusionné

Exportez `model_fused` en ONNX, toujours sans optimisation de l'exportateur, puis comparez les deux graphes : opérateurs, taille du fichier, latence. Pour isoler l'effet de la fusion, les deux sessions sont créées **sans aucune optimisation** d'ONNX Runtime.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : exporter model_fused dans "unet_fusionne.onnx" (optimiser=False)
A_COMPLETER()

for path in ["unet_brut.onnx", "unet_fusionne.onnx"]:
    # ✏️ À COMPLÉTER : latence (médiane, p95) du fichier, avec une session de niveau "aucune"
    med, p95 = A_COMPLETER()
    ops = count_ops(path)
    print(f"{path:20s} {sum(ops.values()):3d} nœuds | {os.path.getsize(path) / 1e6:6.2f} Mo | médiane {med:6.1f} ms | {ops}")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
assert "BatchNormalization" not in count_ops("unet_fusionne.onnx")
_a = make_session("unet_brut.onnx", "aucune").run(None, {"image": test_x[:1]})[0]
_b = make_session("unet_fusionne.onnx", "aucune").run(None, {"image": test_x[:1]})[0]
print(f"OK : 18 nœuds en moins, écart maximal entre les deux graphes {np.abs(_a - _b).max():.1e}")

**Questions.**
1. Le graphe a perdu un quart de ses nœuds. La latence a-t-elle baissé d'un quart ? Expliquez.
2. Refaites la mesure deux ou trois fois : le gain est-il supérieur au bruit de mesure observé à l'exercice 2.2 ?

### L'exportateur le fait déjà

Vous venez d'écrire à la main une optimisation que les outils appliquent seuls. Exportez le modèle d'origine avec les réglages **par défaut** de `export_onnx` (`optimiser=True`) et comptez les opérateurs.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : exporter model (et non model_fused) dans "unet.onnx" avec les réglages par défaut
A_COMPLETER()

count_ops("unet.onnx")

Les BatchNorm ont disparu : en mode évaluation, l'exportateur les fusionne lui-même. L'intérêt de l'avoir fait à la main est de **savoir ce que font les outils**, pour comprendre un graphe exporté, et pour savoir quoi faire quand un outil ne le fait pas.

`unet.onnx` est le fichier que nous utiliserons dans la suite du cours.

## 4. Les optimisations d'ONNX Runtime

Au chargement d'un fichier, ONNX Runtime réécrit le graphe avant de l'exécuter. Le niveau d'optimisation se règle (c'est l'argument `niveau` de `make_session`) :

| Niveau | Ce qu'il fait |
|---|---|
| `aucune` | Exécute le graphe tel quel. |
| `basic` | Simplifications qui conservent le sens du graphe : suppression des nœuds inutiles, calcul à l'avance des constantes, fusion Conv + BatchNorm. |
| `extended` | Fusions plus poussées, propres au matériel visé : par exemple une convolution et sa ReLU en un seul opérateur. |
| `all` | Ajoute les changements d'**organisation mémoire** des tenseurs, adaptés au processeur. |

### Exercice 4.1 — Comparer les niveaux

Partez du graphe **brut** (`unet_brut.onnx`, avec ses BatchNorm) pour voir tout ce qu'ONNX Runtime sait faire. Pour chaque niveau :
- créez une session qui enregistre le graphe optimisé dans un fichier (argument `save_as`) ;
- mesurez sa latence ;
- calculez l'écart maximal entre sa sortie et la sortie de référence `y_ref`.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
x = test_x[:1]
y_ref = make_session("unet_brut.onnx", niveau="aucune").run(None, {"image": x})[0]

resultats = {}
for niveau in NIVEAUX:
    fichier = f"unet_ort_{niveau}.onnx"
    # ✏️ À COMPLÉTER : session de ce niveau à partir de "unet_brut.onnx", qui enregistre le graphe optimisé dans fichier
    s = A_COMPLETER()
    # ✏️ À COMPLÉTER : latence (médiane, p95)
    med, p95 = A_COMPLETER()
    # ✏️ À COMPLÉTER : écart maximal entre la sortie de cette session pour x et y_ref
    ecart = A_COMPLETER()
    resultats[niveau] = med
    print(f"{niveau:9s} médiane {med:6.1f} ms | écart {ecart:.1e} | {count_ops(fichier)}")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
assert set(resultats) == set(NIVEAUX) and all(os.path.exists(f"unet_ort_{n}.onnx") for n in NIVEAUX)
assert "BatchNormalization" not in count_ops("unet_ort_basic.onnx")
print(f"OK : gain de latence entre « aucune » et « all » : {100 * (1 - resultats['all'] / resultats['aucune']):.0f} %")

**Questions.**
1. À quel niveau les BatchNorm disparaissent-elles ? Que deviennent les `Relu` au niveau `extended` ?
2. Au niveau `all` apparaissent des nœuds `ReorderInput` et `ReorderOutput`. Ouvrez `unet_ort_all.onnx` dans Netron : où sont-ils placés ? À quoi servent-ils, selon vous ?
3. Quel niveau apporte le plus gros gain de latence ? Les sorties sont-elles modifiées ?
4. ONNX Runtime avertit que le fichier optimisé au niveau `all` ne doit être utilisé que sur la machine où il a été produit. Pourquoi ?

**À retenir de cette partie.** La même architecture, avec les mêmes poids et le même résultat, peut être nettement plus rapide selon la façon dont le moteur l'exécute. Deux conséquences pour la suite du cours :
- on compare toujours un modèle compressé à la référence **optimisée** (niveau `all`), jamais au graphe brut, sinon on attribue à la compression un gain qui vient du moteur ;
- une optimisation sans perte se vérifie en comparant les sorties, pas en recalculant le mIoU.

## 5. Fiche de référence

La fonction `fiche` rassemble les quatre mesures pour un fichier ONNX. Elle servira dans toutes les séances suivantes : chaque modèle compressé aura sa ligne dans le même tableau.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
def fiche(nom, path, niveau="all", avec_miou=True):
    """Les quatre mesures du cours pour un fichier ONNX (1 fil d'exécution)."""
    session = make_session(path, niveau=niveau)
    med, p95 = latency_ms(session)
    return {
        "modèle": nom,
        "taille (Mo)": round(os.path.getsize(path) / 1e6, 2),
        "latence médiane (ms)": round(med, 1),
        "latence p95 (ms)": round(p95, 1),
        "RAM au pic (Mo)": round(peak_ram_mb(path, niveau), 1),
        "mIoU (%)": round(100 * miou_onnx(session), 2) if avec_miou else None,
    }


def afficher(fiches):
    cles = list(fiches[0])
    print(" | ".join(f"{c:>22s}" if i else f"{c:34s}" for i, c in enumerate(cles)))
    for f in fiches:
        print(" | ".join(f"{str(f[c]):>22s}" if i else f"{f[c]:34s}" for i, c in enumerate(cles)))

### Exercice 5.1 — Le tableau de la séance

Construisez la liste `fiches` avec trois lignes :
1. `"brut, sans optimisation"` : `unet_brut.onnx`, niveau `"aucune"` ;
2. `"fusionné, sans optimisation"` : `unet_fusionne.onnx`, niveau `"aucune"` ;
3. `"référence (ORT all)"` : `unet.onnx`, niveau `"all"`.

Le mIoU est recalculé pour chaque ligne : comptez deux à trois minutes.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : liste des trois fiches
fiches = A_COMPLETER()

afficher(fiches)

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification, puis enregistrement de la référence pour la séance 3
_mious = [f["mIoU (%)"] for f in fiches]
assert len(fiches) == 3 and max(_mious) - min(_mious) < 0.05, \
    "Les trois lignes doivent avoir le même mIoU : ces optimisations sont sans perte."
with open("reference_seance2.json", "w", encoding="utf-8") as f:
    json.dump(fiches[-1], f, ensure_ascii=False, indent=2)
print("OK : reference_seance2.json enregistré. À conserver avec unet.onnx et unet_camvid_base32.pt.")

# Dans Colab, pour télécharger les fichiers :
# from google.colab import files; files.download("unet.onnx"); files.download("reference_seance2.json")

**Question.** Comparez les trois lignes, colonne par colonne. Qu'avons-nous gagné aujourd'hui ? Qu'est-ce qui n'a pas changé ? Une grandeur s'est-elle dégradée ? Lesquelles les séances suivantes devront-elles attaquer ?

## À retenir

- Un fichier **ONNX** est un graphe d'opérateurs standard et ses poids : il s'exécute sans PyTorch, sur des moteurs différents selon la cible.
- Une latence se mesure après échauffement, sur plusieurs répétitions, et se rapporte en médiane et p95, avec le matériel et le nombre de fils. Une différence de quelques pour cent est du bruit.
- La **taille du fichier** suit le nombre de paramètres ; le **pic de mémoire vive** suit les activations. Ce sont deux problèmes distincts.
- À l'inférence, une **BatchNorm se fusionne exactement dans la convolution** qui la précède. Les outils le font seuls ; vous savez maintenant pourquoi c'est correct.
- Le moteur d'inférence compte autant que le modèle : on compare toujours à une référence optimisée.

## Pour la séance 3

Jusqu'ici, tout était sans perte. La **quantification** remplace les nombres `float32` par des entiers sur 8 bits : un fichier quatre fois plus petit, des calculs plus rapides, et pour la première fois une précision qui peut baisser. Il faudra apprendre à mesurer cette baisse, classe par classe, et à la limiter.

D'ici là : conservez `unet.onnx`, `reference_seance2.json` et votre fichier `unet_camvid_base32.pt`.